In [ ]:
# print("Hellow!")

Hellow!


In [1]:
import os
import json
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
from google import genai
from google.genai import types
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [4]:
from ingest import load_faq_data, build_index

In [5]:
from rag_helper import RAGBase

In [13]:
model='gemini-2.5-flash'

In [7]:
documents = load_faq_data()
index = build_index(documents)

In [8]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index = index,
    llm_client = client,
    instructions = instructions
)

In [28]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

AttributeError: 'RAGBase' object has no attribute 'chat_session'

In [15]:
messages = [
    types.Content(
        role="user",
        parts=[types.Part.from_text(text="I just discovered the course. Can I join it?")]
    )
]

config = types.GenerateContentConfig(
    temperature=0.2, # Controls randomness (lower is more deterministic)
    max_output_tokens=800   
)

response = client.models.generate_content(
        model=model,
        contents=messages,
        config=config
)

response.text

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [29]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [ ]:
search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search",
            description="Search the FAQ database for entries matching the given query.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "query": types.Schema(
                        type=types.Type.STRING,
                        description="Search query text to look up in the course FAQ."
                    )
                },
                required=["query"]
            )
        )
    ]
)

In [ ]:
tool_config = types.GenerateContentConfig(
    tools=[search_tool],
    temperature=0.2
)

In [ ]:
response = client.models.generate_content(
    model=model,
    contents=messages,
    config=tool_config # Pass tools here
)

TypeError: Models.generate_content() got an unexpected keyword argument 'tools'

In [ ]:
response.text

NameError: name 'response' is not defined

In [ ]:
len(response.text)

In [ ]:
call = response.outputs[0]

In [ ]:
print(call)

In [ ]:
arg = json.loads(call.arguments_json)

In [ ]:
call.name

In [ ]:
result = search(arg['query'])

In [ ]:
result_json = json.dumps(result.to_dict())

In [ ]:
# print(result_json)

In [ ]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json
}

In [ ]:
messages.append(call)

In [ ]:
print(messages)

In [ ]:
# Extract token usage from metadata
usage = response.usage_metadata
prompt_tokens = usage.prompt_token_count
candidate_tokens = usage.candidates_token_count

# Print individual metrics
print(f"Prompt (Input) Tokens: {prompt_tokens}")
print(f"Candidates (Output) Tokens: {candidate_tokens}")
print(f"Total Tokens Used: {usage.total_token_count}")

In [ ]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(prompt_tokens, candidate_tokens)

print("Total Cost: $", round(result["total_cost"], 8))